# Study report: clock-drift

The study asks why the kernel cycle counter loses cycles against the
stimulus timer TIM2 (the same clock), and if the loss changes the
latency data. It tests two causes: the cycles that each SysTick
reprogram loses (H1), and whole reloads that the SysTick driver loses
with 256 load timers (H2). The tree `ws-up-driftfix` adds the option
`CORTEX_M_SYSTICK_RELOAD_GAP_COMP`: the driver measures the uncounted
cycles of each reprogram with the DWT cycle counter and adds them to
the software count.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-clock-drift"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The drift of each point

The values come from the `LAT META` keys of each run. `clock_drift_cyc`
is the number of cycles that `k_cycle_get_64()` fell behind the stimulus
timer since `stim_init()`. A negative value means that it ran ahead.
`clock_elapsed_cyc` gives the elapsed cycles of `k_cycle_get_64()` for
the same time. `drift/wake` is `clock_drift_cyc` / samples (one wake per
sample). `ppm` is `clock_drift_cyc` / `clock_elapsed_cyc` x 1e6.
`wake-clock-drift` and `wake-clock-drift-layout` are imports from the
study `zephyr-versions`.

In [ ]:
EXPERIMENTS = ["wake-clock-drift", "wake-clock-drift-fix", "wake-clock-drift-layout"]
drift = inst.drift([e for e in EXPERIMENTS if nbfig.built(inst, e)]).rename(
    columns={"drift_per_sample": "drift/wake", "drift_ppm": "ppm"})
drift.reindex(columns=["experiment", "tree", "zephyr", "point", "samples", "load_timers", "clock_elapsed_cyc",
                       "clock_drift_cyc", "drift/wake", "ppm"])

In [ ]:
import plotly.graph_objects as go


def bars(df, x, series, y, title, xtitle, ytitle, hline=None):
    """Grouped bars: one colour for each series, a text label on each bar. hline: (y, text) as a dashed line."""
    fig = go.Figure()
    for k, s in enumerate(dict.fromkeys(df[series])):
        sub = df[df[series] == s]
        fig.add_trace(go.Bar(x=sub[x], y=sub[y], name=str(s), marker_color=nbfig.color(k),
                             text=[f"{v:.2f}" for v in sub[y]], textposition="outside", customdata=sub["point"],
                             hovertemplate="%{customdata}: %{y:.3f}<extra>" + str(s) + "</extra>"))
    if hline is not None:
        fig.add_hline(y=hline[0], line_dash="dash", line_color="#52514c")
        # the text of the line is a legend entry, so that it does not cover a bar
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=hline[1],
                                 line={"dash": "dash", "color": "#52514c"}))
    return nbfig.style(fig, title, xtitle, ytitle, barmode="group")

## H1 a: the drift and the number of SysTick reprograms (wake-clock-drift)

The anchors and the load timers change the number of SysTick reprograms
per wake. With `default` (time slicing), each wake reprograms SysTick for
the slice timeout, unless an earlier timeout is in the queue. `noslice`
(`TIMESLICING=n`) has no slice timeout. `switch` is the other arch
interface for the context switch. With `t16` (16 load timers), the slice
timeout is seldom the first timeout, thus fewer reprograms occur. The
dashed line is about 8 cycles: the loss of one reprogram (H1). The
design gives no number of reprograms for each point, so a person gives
the verdict.

In [ ]:
d = drift[drift.experiment == "wake-clock-drift"].copy()
if len(d):
    d["anchor"] = d.point.str.split("-").str[0]
    d["timers"] = d.point.str.split("-").str[1]
    bars(d, "anchor", "timers", "drift/wake", "wake-clock-drift: cycles lost per wake", "anchor",
         "clock_drift_cyc / samples", hline=(8, "about 8 cycles: one reprogram (H1)")).show()
    display(d.pivot_table(index="anchor", columns="timers", values="drift/wake", sort=False))

## H1 b: the gap compensation (wake-clock-drift-fix)

The cube `comp` x `anchor` on the tree `ws-up-driftfix`. `nocomp` is the
tree without the option: the control with the drift. `gapcomp`
(`CORTEX_M_SYSTICK_RELOAD_GAP_COMP=y`) is the fix of the first cause.
H1 b expects a drift close to 0 for `gapcomp`. The design gives no number
for "close to 0", so a person gives the verdict.

In [ ]:
d = drift[(drift.experiment == "wake-clock-drift-fix") & ~drift.point.str.startswith("t256")].copy()
if len(d):
    d["comp"] = d.point.str.split("-").str[0]
    d["anchor"] = d.point.str.split("-").str[1]
    fig = bars(d, "anchor", "comp", "ppm", "wake-clock-drift-fix: drift with and without the gap compensation",
               "anchor", "ppm of the elapsed cycles")
    fig.update_traces(texttemplate="%{y:.1f}")
    fig.add_hline(y=0, line_color="#52514c", line_width=1)
    fig.show()
    display(d.pivot_table(index="anchor", columns="comp", values=["ppm", "drift/wake"], sort=False))

## H2 a: whole reloads with 256 load timers (wake-clock-drift-fix)

The driver programs a reload of 10 us or more (`min_delay`) and sees one
counter wrap or less. With 256 periodic timers, the kernel can mask the
SysTick IRQ for longer than the shortest reload, because each timeout
insert walks the list under the lock. Then the driver loses whole
reloads. The three points `t256-*` have the gap compensation, and they
change the shortest reload (`zephyr,min-timeout-cycles`): `min10us` is
the default of the driver; `min50us` and `min200us` use the overlays in
`common/boards/nucleo_g474re/systick/`. The dashed line is the point
`gapcomp-default` (no load timers). H2 a expects that `min50us` and
`min200us` do not have the loss of `min10us`; a person gives the
verdict.

In [ ]:
d = drift[(drift.experiment == "wake-clock-drift-fix") & drift.point.str.startswith("t256")].copy()
if len(d):
    d["min_reload"] = d.point.str.split("-").str[1]
    ref = drift[(drift.experiment == "wake-clock-drift-fix") & (drift.point == "gapcomp-default")]
    fig = bars(d, "min_reload", "tree", "ppm", "wake-clock-drift-fix: drift with 256 load timers, by shortest reload",
               "shortest SysTick reload", "ppm of the elapsed cycles",
               hline=(ref.ppm.iloc[0], "gapcomp-default (no load timers)") if len(ref) else None)
    fig.update_traces(texttemplate="%{y:.0f}")
    fig.show()
    display(pd.concat([d, ref])[["point", "load_timers", "samples", "clock_drift_cyc", "drift/wake", "ppm"]])

## The histogram size and the sample count (wake-clock-drift-layout)

The same wake (semaphore, time slicing, no load timers) gave different
drift values in `wake-clock-drift` and in `wake-timers-1m` (point
`t0-default`). The two runs have a different histogram size, and thus a
different code and data layout. They also have a different number of
samples. This experiment changes these two factors: `hdefault` (the
default bins of the app) or `hcoarse` (2048 fine and 2048 coarse bins),
and 200000 (`s200k`) or 1000000 (`s1m`) samples. No prediction came
before the run; the values are the central result `layout-drift`.

In [ ]:
d = drift[drift.experiment == "wake-clock-drift-layout"].copy()
if len(d):
    d["bins"] = d.point.str.split("-").str[0]
    d["sample_count"] = d.point.str.split("-").str[1]
    bars(d, "bins", "sample_count", "drift/wake", "wake-clock-drift-layout: cycles lost per wake",
         "histogram size", "clock_drift_cyc / samples").show()
    display(d.pivot_table(index="bins", columns="sample_count", values="drift/wake", sort=False))

## Does the drift change the latency data?

`stim_arm()` reads the stimulus timer and `k_cycle_get_64()` again for
each edge, and calculates the tick phase from this fresh pair (guide:
"The kernel cycle counter loses cycles"). The figure shows the
latencies of the cube of `wake-clock-drift-fix`, with (`gapcomp`) and
without (`nocomp`) the gap compensation, for each anchor. The colour is
the value of `comp`, the symbol is the statistic.

In [ ]:
if nbfig.built(inst, "wake-clock-drift-fix"):
    df = inst.rows("wake-clock-drift-fix")
    cube = df[df["axis.timers"] == "t0"]
    nbfig.dots(cube, "wake-clock-drift-fix: latency with and without the gap compensation", x="axis.anchor",
               group="axis.comp", stats=("p50", "p99", "max")).update_xaxes(title="anchor").show()
    display(cube.pivot_table(index="point", columns="metric", values=["p50", "p99", "max"], aggfunc="first",
                             sort=False))